[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bethavila17/dncp-riesgo-contrataciones/blob/main/notebooks/03_modelos_y_evaluacion.ipynb)

# 3. Modelos: medición cuantitativa y elección del mejor algoritmo
**Proyecto III Big Data — Maestría en Ciencia de Datos (UCOM)** · Bethania Avila, Arturo Fernández, Gustavo Bogado

**Métricas definidas antes de entrenar** (no existe una etiqueta oficial de irregularidad; los casos de referencia son los procesos con protesta, denuncia o investigación):

| Métrica | Qué mide | Por qué |
|---|---|---|
| **AUC-PR** (principal) | área bajo la curva precisión–exhaustividad | adecuada con clases escasas (4–8 % de casos) |
| Precision@k / Lift@k | aciertos entre los k procesos de mayor puntaje / mejora sobre el azar | el auditor revisa k procesos |
| Recall y F1 al 5 % superior | cobertura con capacidad de revisión fija | simula la capacidad de auditoría |
| ROC-AUC | capacidad general de ordenar | complementaria |
| Estabilidad (Spearman, Jaccard del 5 % superior) | coincidencia al reentrenar con submuestras | un ranking que cambia no sirve para auditar |
| Banderas en el 5 % superior | concentración de indicadores de riesgo | validación interna |
| Costo | segundos de entrenamiento, ms por 1.000 procesos | relevante en Big Data |

Para los detectores, la métrica principal se calcula en procesos con **2 o más oferentes** (controla que las protestas crecen con la competencia).

**Regla de decisión:** (a) se descartan los detectores inestables (Jaccard < 0,5); (b) gana el mayor AUC-PR; (c) si la diferencia es < 0,02, gana el de mayor Precision@100; (d) si persiste el empate, el de menor costo.

**Validación temporal:** entrenamiento 2021–2024, prueba 2025, puntuación de procesos nuevos 2026.

In [ ]:
# Configuración del entorno (Google Colab o local)
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/bethavila17/dncp-riesgo-contrataciones.git"  # <- URL del repositorio en GitHub
USAR_DRIVE = False  # True: guarda el data lake en Google Drive y no repite descargas entre sesiones

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    RAIZ = Path("/content/dncp-riesgo-contrataciones")
    if not RAIZ.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(RAIZ)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(RAIZ / "requirements-colab.txt")], check=True)
    if USAR_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        os.environ["DNCP_DATA_DIR"] = "/content/drive/MyDrive/dncp_data_lake"
else:
    RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ / "src"))

import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dncp_riesgo import graficos as gr
from dncp_riesgo.pipeline import asegurar_datos, preparar

cfg, lake = preparar()
gr.estilo()
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)
print("Entorno:", "Google Colab" if EN_COLAB else "local", "| años:", cfg["fuentes"]["ocp_registry"]["anios"])

In [ ]:
asegurar_datos(lake, cfg, hasta="gold")
from dncp_riesgo.modelos import entrenar_y_puntuar, CRI_NOMBRE
res = entrenar_y_puntuar(lake, cfg)
pd.Series(res.tiempos, name="segundos").to_frame()

## 3.1 Detectores de anomalías: comparación cuantitativa

In [ ]:
print("Tasa base (≥2 oferentes):", round(res.evaluacion_detectores.attrs.get("tasa_base_competitivos", np.nan), 4))
res.evaluacion_detectores

In [ ]:
print("Detector elegido:", res.artefactos.nombre_detector); print("Motivo:", res.decision_detector)

In [ ]:
from sklearn.metrics import precision_recall_curve
te = res.prueba; d = res.datos
comp = te[(d.loc[te, "n_oferentes"] >= 2).to_numpy()]
y = d.loc[comp, "y_queja"].to_numpy()
fig, ax = plt.subplots(figsize=(6, 3.6))
nombres = ["Isolation Forest global", "Isolation Forest por modalidad", "Autoencoder"]
for nombre, color in zip(nombres, gr.SERIES):
    pr, rc, _ = precision_recall_curve(y, res.puntajes_detectores[nombre][comp])
    ax.plot(rc, pr, color=color, lw=2, label=nombre)
ax.axhline(y.mean(), ls=":", color=gr.TENUE, lw=1)
ax.text(0.98, y.mean(), "tasa base", ha="right", va="bottom", fontsize=8, color=gr.TINTA_2)
ax.set_xlabel("exhaustividad (recall)"); ax.set_ylabel("precisión"); ax.legend()
ax.set_title("Curvas precisión–exhaustividad, prueba 2025 (≥2 oferentes)")
gr.guardar(fig, "fig07_pr_detectores"); plt.show()

In [ ]:
est = res.evaluacion_detectores.loc[nombres + [CRI_NOMBRE], ["Jaccard top 5%", "Spearman entre reentrenamientos"]]
fig, ax = plt.subplots(figsize=(6, 2.6))
ax.barh(est.index[::-1], est["Jaccard top 5%"][::-1], color=gr.SERIES[0], height=0.6)
ax.axvline(cfg["modelos"]["jaccard_minimo"], ls="--", color=gr.NIVELES["Alto"], lw=1)
ax.set_xlim(0, 1); ax.set_xlabel("Jaccard del 5 % superior entre reentrenamientos")
ax.grid(axis="x"); ax.grid(axis="y", visible=False); ax.set_title("Estabilidad de los rankings")
gr.guardar(fig, "fig08_estabilidad"); plt.show()
est

### Concentración de banderas rojas y perfil del 5 % superior de cada método

In [ ]:
res.concentracion

## 3.2 Validación de las banderas que forman el índice de reglas (CRI)

In [ ]:
res.validacion_banderas

## 3.3 Riesgo de impugnación (supervisado) frente a líneas base
Mide conflictividad, no irregularidad: se informa aparte del riesgo de integridad.

In [ ]:
print("Modelo elegido:", res.artefactos.nombre_supervisado, "—", res.decision_supervisado)
res.evaluacion_supervisados

In [ ]:
import shap
from dncp_riesgo.modelos import ExplicadorSHAP, matriz, VARIABLES_SUPERVISADO
muestra = d.loc[te].sample(min(3000, len(te)), random_state=0)
exp = ExplicadorSHAP(res.artefactos.supervisado).valores(matriz(muestra, VARIABLES_SUPERVISADO))
shap.plots.beeswarm(exp, max_display=12, show=False)
fig = plt.gcf(); fig.set_size_inches(8, 5); plt.title("SHAP: riesgo de impugnación (prueba 2025)")
gr.guardar(fig, "fig09_shap_impugnacion"); plt.show()

## 3.4 Explicación de las anomalías del detector elegido
Variables que más pesan en la anomalía de los procesos de mayor prioridad.

In [ ]:
top = res.puntajes.query("periodo == 'prueba' and factores_anomalia != ''")
pd.Series([v for t in top["factores_anomalia"] for v in t.split("; ")]).value_counts().head(10).rename("veces entre los 3 factores principales").to_frame()

## 3.5 Prioridad de auditoría: perfil del 5 % superior (2025)

In [ ]:
res.perfil_prioridad

In [ ]:
cols = ["prioridad", "puntaje_integridad", "pct_impugnacion", "comprador_nombre", "proveedor_nombre", "metodo_grupo", "monto_adjudicado", "banderas_activas", "factores_anomalia"]
res.puntajes.query("periodo == 'prueba'").sort_values("prioridad", ascending=False)[cols].head(15)